<a href="https://colab.research.google.com/github/carlavilla/DataViz2025/blob/main/co-occurence%20test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

see
https://plotly.com/python/network-graphs/

and others here
https://www.google.com/search?q=python+network+visulization&rlz=1C1GCEA_enUS1158US1158&oq=python+network+visulization&gs_lcrp=EgZjaHJvbWUyBggAEEUYOTIJCAEQABgNGIAEMggIAhAAGBYYHjIICAMQABgWGB4yCAgEEAAYFhgeMgoIBRAAGAgYDRgeMgoIBhAAGAgYDRgeMgoIBxAAGAgYDRgeMgoICBAAGAgYDRgeMgoICRAAGAgYDRge0gEINjUxM2owajeoAgCwAgA&sourceid=chrome&ie=UTF-8


In [1]:
import urllib
from google.colab import files
import seaborn as sns
from google.colab import data_table
data_table.enable_dataframe_formatter()
import time, os, sys, re
import zipfile, json, datetime, string
import numpy as np
from statistics import *
import matplotlib.pyplot as plt
import pandas as pd
import pandas_datareader as pdr
from pandas_datareader import wb
from pandas.io.formats.style import Styler
import networkx as nx
from itertools import combinations

In [2]:
urllib.request.urlretrieve("https://docs.google.com/uc?id=1IcgJnGWdt6_XZbRYSpyrkQSKj92NoFXo&export=download", "IHNCHNA2025_CommunitySurvey_CompleteSet_Nov7.24.xlsx")
chna = pd.read_excel('IHNCHNA2025_CommunitySurvey_CompleteSet_Nov7.24.xlsx')

In [3]:
chna2 = chna.filter(regex='A6_', axis=1)

In [8]:
chna2.columns

Index(['A6_1', 'A6_2', 'A6_3', 'A6_4', 'A6_5', 'A6_6', 'A6_7', 'A6_8', 'A6_9',
       'A6_10', 'A6_11', 'A6_12', 'A6_13', 'A6_14', 'A6_15', 'A6_16', 'A6_17',
       'A6_18', 'A6_19', 'A6_20', 'A6_21'],
      dtype='object')

In [7]:
chna2 = chna2.drop(columns=['A6_22', 'A6_22_TEXT'])

KeyError: "['A6_22', 'A6_22_TEXT'] not found in axis"

In [24]:


def create_cooccurrence_graph(data):
    """
    Creates a co-occurrence graph from a list of lists.

    Args:
        data: A list of lists representing the data.

    Returns:
        A networkx graph object.
    """
    graph = nx.Graph()
    for row in data:
         for x, y in combinations(row, 2):
            if graph.has_edge(x, y):
                graph[x][y]['weight'] += 1
            else:
                graph.add_edge(x, y, weight=1)
    return graph

def visualize_cooccurrence_graph(graph):
    """
    Visualizes a co-occurrence graph.

    Args:
        graph: A networkx graph object.
    """
    plt.figure(figsize=(10, 8))
    pos = nx.spring_layout(graph)
    nx.draw(graph, pos, with_labels=True, node_color='skyblue', node_size=1500, font_size=10, width=[d['weight'] for (u,v,d) in graph.edges(data=True)])
    edge_labels = {(u, v): d['weight'] for u, v, d in graph.edges(data=True)}
    nx.draw_networkx_edge_labels(graph, pos, edge_labels=edge_labels)
    plt.title("Co-occurrence Graph")
    plt.show()

# Example usage:
#data = [
    #['A', 'B', 'C'],
    #['B', 'C', 'D'],
    #['A', 'B'],
    #['C', 'D']
#]

graph = create_cooccurrence_graph(chna2)
visualize_cooccurrence_graph(graph)

UnboundLocalError: cannot access local variable 'y' where it is not associated with a value

In [14]:
import plotly.graph_objects as go

import networkx as nx

G = nx.Graph(chna2)

NetworkXError: Input is not a correct Pandas DataFrame edge-list.

In [11]:
edge_x = []
edge_y = []
for edge in G.edges():
    x0, y0 = G.nodes[edge[0]]['pos']
    x1, y1 = G.nodes[edge[1]]['pos']
    edge_x.append(x0)
    edge_x.append(x1)
    edge_x.append(None)
    edge_y.append(y0)
    edge_y.append(y1)
    edge_y.append(None)

edge_trace = go.Scatter(
    x=edge_x, y=edge_y,
    line=dict(width=0.5, color='#888'),
    hoverinfo='none',
    mode='lines')

node_x = []
node_y = []
for node in G.nodes():
    x, y = G.nodes[node]['pos']
    node_x.append(x)
    node_y.append(y)

node_trace = go.Scatter(
    x=node_x, y=node_y,
    mode='markers',
    hoverinfo='text',
    marker=dict(
        showscale=True,
        # colorscale options
        #'Greys' | 'YlGnBu' | 'Greens' | 'YlOrRd' | 'Bluered' | 'RdBu' |
        #'Reds' | 'Blues' | 'Picnic' | 'Rainbow' | 'Portland' | 'Jet' |
        #'Hot' | 'Blackbody' | 'Earth' | 'Electric' | 'Viridis' |
        colorscale='YlGnBu',
        reversescale=True,
        color=[],
        size=10,
        colorbar=dict(
            thickness=15,
            title=dict(
              text='Node Connections',
              side='right'
            ),
            xanchor='left',
        ),
        line_width=2))

In [12]:
node_adjacencies = []
node_text = []
for node, adjacencies in enumerate(G.adjacency()):
    node_adjacencies.append(len(adjacencies[1]))
    node_text.append('# of connections: '+str(len(adjacencies[1])))

node_trace.marker.color = node_adjacencies
node_trace.text = node_text

In [13]:
fig = go.Figure(data=[edge_trace, node_trace],
             layout=go.Layout(
                title=dict(
                    text="<br>Network graph made with Python",
                    font=dict(
                        size=16
                    )
                ),
                showlegend=False,
                hovermode='closest',
                margin=dict(b=20,l=5,r=5,t=40),
                annotations=[ dict(
                    text="Python code: <a href='https://plotly.com/python/network-graphs/'> https://plotly.com/python/network-graphs/</a>",
                    showarrow=False,
                    xref="paper", yref="paper",
                    x=0.005, y=-0.002 ) ],
                xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                yaxis=dict(showgrid=False, zeroline=False, showticklabels=False))
                )
fig.show()

In [21]:
chna2.head()

,A6_1,A6_2,A6_3,A6_4,A6_5,A6_6,A6_7,A6_8,A6_9,A6_10,...,A6_12,A6_13,A6_14,A6_15,A6_16,A6_17,A6_18,A6_19,A6_20,A6_21
0,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,...,NaN,NaN,1.0,NaN,NaN,1.0,NaN,1.0,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,...,NaN,NaN,1.0,NaN,NaN,1.0,NaN,1.0,NaN,NaN
2,NaN,NaN,NaN,1.0,NaN,NaN,1.0,NaN,NaN,NaN,...,NaN,NaN,1.0,NaN,NaN,1.0,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,1.0,1.0,NaN,NaN,NaN,NaN,...,NaN,NaN,1.0,1.0,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
def bunch(data):
    """
    creating list of bunches
    """
list = []
  for row in data:

            if graph.has_edge(x, y):
                graph[x][y]['weight'] += 1
            else:
                graph.add_edge(x, y, weight=1)
    return graph

In [33]:
for row in chna2:
  if chna2.item()=='1.0':
    print [column]

AttributeError: 'DataFrame' object has no attribute 'item'